# 00 — Raw data validation

This notebook validates the 24 raw monthly CSV files in `data/raw/csv/`. It reports what is present and flags quality issues, but it never cleans, rewrites, or saves the raw data.

Run this notebook in the project's Spark environment. The Spark master and workers are supplied by that environment; the notebook does not install PySpark or start a cluster.

## 1. Setup and safe path resolution

The code supports common project working locations. The resolved project location is used internally only. Any path shown to the reader remains project-relative and begins with `data/...`.

In [1]:
from pathlib import Path
import csv
import re

from pyspark import StorageLevel
from pyspark.sql import SparkSession, functions as F

def find_project_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data/raw/csv").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate data/raw/csv/. Run this notebook inside the Airline project.")

PROJECT_ROOT = find_project_root()
DATA_REL = Path("data/raw/csv")
DATA_DIR = PROJECT_ROOT / DATA_REL

spark = SparkSession.builder.appName("airline-00-data-validation").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("Spark session ready.")
print(f"Raw input: {DATA_REL.as_posix()}/")

Spark session ready.
Raw input: data/raw/csv/


## 2. File inventory and month coverage

A complete delivery contains one CSV for every month of 2023 and 2024. The check below uses filenames only and shows relative source paths.

In [2]:
csv_files = sorted(DATA_DIR.glob("*.csv"))
relative_files = [f"{DATA_REL.as_posix()}/{path.name}" for path in csv_files]

print(f"Monthly CSV files found: {len(csv_files)}")
for relative_path in relative_files:
    print(f"  {relative_path}")

filename_pattern = re.compile(r"_(2023|2024)_(\d{1,2})\.csv$")
observed_months = {
    (int(match.group(1)), int(match.group(2)))
    for path in csv_files
    if (match := filename_pattern.search(path.name))
}
expected_months = {(year, month) for year in (2023, 2024) for month in range(1, 13)}
missing_months = sorted(expected_months - observed_months)
unexpected_months = sorted(observed_months - expected_months)

print(f"Expected monthly files: {len(expected_months)}")
print(f"Missing months: {missing_months or 'none'}")
print(f"Unexpected months: {unexpected_months or 'none'}")
assert len(csv_files) == 24, f"Expected 24 CSV files in data/raw/csv/, found {len(csv_files)}."
assert observed_months == expected_months, "Filename month coverage is incomplete or unexpected."
print("PASS: all 24 expected monthly files are present.")

Monthly CSV files found: 24
  data/raw/csv/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2023_1.csv
  data/raw/csv/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2023_10.csv
  data/raw/csv/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2023_11.csv
  data/raw/csv/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2023_12.csv
  data/raw/csv/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2023_2.csv
  data/raw/csv/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2023_3.csv
  data/raw/csv/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2023_4.csv
  data/raw/csv/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2023_5.csv
  data/raw/csv/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2023_6.csv
  data/raw/csv/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2023_7.csv
  data/raw/csv/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2023_8.csv
  data/raw/c

If both assertions pass, the delivery has exactly the expected number of files and every required year/month appears once in the filename inventory.

## 3. Header and schema consistency

Before Spark combines the monthly files, compare their raw header rows. This catches missing, added, or reordered columns that a unioned read could otherwise hide.

In [3]:
headers = {}
for path in csv_files:
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        headers[path.name] = tuple(next(csv.reader(handle)))

reference_name = csv_files[0].name
reference_header = headers[reference_name]
inconsistent_files = [name for name, header in headers.items() if header != reference_header]
header_differences = {
    name: {
        "missing_vs_reference": sorted(set(reference_header) - set(header)),
        "added_vs_reference": sorted(set(header) - set(reference_header)),
        "same_columns_different_order": set(header) == set(reference_header) and header != reference_header,
    }
    for name, header in headers.items()
    if header != reference_header
}

required_columns = {
    "Year", "Month", "DayofMonth", "DayOfWeek", "FlightDate",
    "Reporting_Airline", "Flight_Number_Reporting_Airline",
    "Origin", "Dest", "CRSDepTime", "CRSArrTime",
    "Cancelled", "Diverted", "ArrDelay", "ArrDelayMinutes",
    "CarrierDelay", "WeatherDelay", "NASDelay",
    "SecurityDelay", "LateAircraftDelay",
}
missing_required = sorted(required_columns - set(reference_header))

print(f"Columns in reference header: {len(reference_header)}")
print(f"Files with a different header: {len(inconsistent_files)}")
for name in inconsistent_files:
    print(f"  {DATA_REL.as_posix()}/{name}: {header_differences[name]}")
print(f"Missing required columns: {missing_required or 'none'}")

assert not inconsistent_files, "Monthly CSV headers are not identical."
assert not missing_required, f"Required columns are missing: {missing_required}"
print("PASS: all monthly headers are identical and required columns are present.")

Columns in reference header: 110
Files with a different header: 0
Missing required columns: none
PASS: all monthly headers are identical and required columns are present.


Spark now reads all raw files together. `inferSchema` is intentionally used on the source as delivered; no cleaned copy is created. The DataFrame is persisted temporarily because several validation actions follow.

In [4]:
flights = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("mode", "FAILFAST")
    .csv(str(DATA_DIR / "*.csv"))
    .persist(StorageLevel.DISK_ONLY)
)

row_count = flights.count()
print(f"Rows loaded: {row_count:,}")
print(f"Spark columns: {len(flights.columns)}")
flights.printSchema()

Rows loaded: 13,926,960
Spark columns: 110
root
 |-- Year: integer (nullable = true)
 |-- Quarter: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- FlightDate: date (nullable = true)
 |-- Reporting_Airline: string (nullable = true)
 |-- DOT_ID_Reporting_Airline: integer (nullable = true)
 |-- IATA_CODE_Reporting_Airline: string (nullable = true)
 |-- Tail_Number: string (nullable = true)
 |-- Flight_Number_Reporting_Airline: integer (nullable = true)
 |-- OriginAirportID: integer (nullable = true)
 |-- OriginAirportSeqID: integer (nullable = true)
 |-- OriginCityMarketID: integer (nullable = true)
 |-- Origin: string (nullable = true)
 |-- OriginCityName: string (nullable = true)
 |-- OriginState: string (nullable = true)
 |-- OriginStateFips: integer (nullable = true)
 |-- OriginStateName: string (nullable = true)
 |-- OriginWac: integer (nullable = true)
 |-- DestAirportID: intege

## 4. Row counts, dates, and data coverage

Filename coverage does not guarantee that the rows contain the correct dates. The next checks summarize the actual flight records.

In [5]:
dated_flights = flights.withColumn("_flight_date", F.to_date("FlightDate"))

date_summary = dated_flights.agg(
    F.count("*").alias("row_count"),
    F.min("_flight_date").alias("min_flight_date"),
    F.max("_flight_date").alias("max_flight_date"),
    F.countDistinct("_flight_date").alias("distinct_flight_dates"),
)
date_summary.show(truncate=False)

monthly_counts = (
    dated_flights
    .groupBy(F.year("_flight_date").alias("year"), F.month("_flight_date").alias("month"))
    .agg(F.count("*").alias("rows"))
    .orderBy("year", "month")
)
monthly_counts.show(24, truncate=False)

data_month_count = monthly_counts.count()
out_of_range_rows = dated_flights.filter(
    (F.col("_flight_date") < F.lit("2023-01-01").cast("date"))
    | (F.col("_flight_date") > F.lit("2024-12-31").cast("date"))
    | F.col("_flight_date").isNull()
).count()
print(f"Distinct year/month combinations in row data: {data_month_count}")
print(f"Rows with null or out-of-range FlightDate: {out_of_range_rows:,}")
assert data_month_count == 24, "Row data does not cover exactly 24 year/month combinations."
assert out_of_range_rows == 0, "Some rows have null or unexpected FlightDate values."

+---------+---------------+---------------+---------------------+
|row_count|min_flight_date|max_flight_date|distinct_flight_dates|
+---------+---------------+---------------+---------------------+
|13926960 |2023-01-01     |2024-12-31     |731                  |
+---------+---------------+---------------+---------------------+



+----+-----+------+
|year|month|rows  |
+----+-----+------+
|2023|1    |538837|
|2023|2    |502749|
|2023|3    |580322|
|2023|4    |561441|
|2023|5    |579958|
|2023|6    |577262|
|2023|7    |601866|
|2023|8    |602987|
|2023|9    |569338|
|2023|10   |598968|
|2023|11   |563777|
|2023|12   |570394|
|2024|1    |547271|
|2024|2    |519221|
|2024|3    |591767|
|2024|4    |582185|
|2024|5    |609743|
|2024|6    |611132|
|2024|7    |634613|
|2024|8    |619025|
|2024|9    |582622|
|2024|10   |615497|
|2024|11   |575404|
|2024|12   |590581|
+----+-----+------+



Distinct year/month combinations in row data: 24
Rows with null or out-of-range FlightDate: 0


The monthly table makes unusually small or large deliveries visible. The assertions also ensure all parsed flight dates fall inside the intended two-year window.

## 5. Airline coverage

`Reporting_Airline` is the operating carrier code used for reporting. The complete distribution verifies whether the source contains multiple airlines rather than only `9E`.

In [6]:
airline_counts = (
    flights
    .groupBy("Reporting_Airline")
    .agg(F.count("*").alias("rows"))
    .orderBy(F.desc("rows"), F.asc_nulls_last("Reporting_Airline"))
)
airline_counts.show(100, truncate=False)

distinct_airlines = flights.select("Reporting_Airline").where(F.col("Reporting_Airline").isNotNull()).distinct().count()
non_9e_rows = flights.filter(F.col("Reporting_Airline") != "9E").count()
print(f"Distinct non-null airlines: {distinct_airlines}")
print(f"Rows belonging to airlines other than 9E: {non_9e_rows:,}")
assert distinct_airlines > 1, "Dataset contains only one reporting airline."
assert non_9e_rows > 0, "Dataset contains only 9E rows."
print("PASS: the dataset contains multiple airlines and is not limited to 9E.")

+-----------------+-------+
|Reporting_Airline|rows   |
+-----------------+-------+
|WN               |2857884|
|DL               |1994160|
|AA               |1924837|
|UA               |1492663|
|OO               |1419821|
|YX               |596740 |
|NK               |524974 |
|B6               |515134 |
|MQ               |507443 |
|AS               |491163 |
|OH               |422115 |
|9E               |401614 |
|F9               |386166 |
|G4               |232749 |
|HA               |159497 |
+-----------------+-------+



Distinct non-null airlines: 15
Rows belonging to airlines other than 9E: 13,525,346
PASS: the dataset contains multiple airlines and is not limited to 9E.


## 6. Duplicate checks

Two definitions are useful. An **exact-row duplicate** repeats every source value. A **business-key duplicate** repeats the scheduled flight identity and may still differ in another field. Both checks require a shuffle, so expect this section to be more expensive than simple counts.

In [7]:
distinct_row_count = flights.distinct().count()
exact_duplicate_rows = row_count - distinct_row_count
print(f"Exact duplicate rows: {exact_duplicate_rows:,}")

Exact duplicate rows: 0


In [8]:
flight_key = [
    "FlightDate", "Reporting_Airline", "Flight_Number_Reporting_Airline",
    "Origin", "Dest", "CRSDepTime",
]
duplicate_key_groups = (
    flights.groupBy(*flight_key)
    .agg(F.count("*").alias("rows_per_key"))
    .filter(F.col("rows_per_key") > 1)
)
duplicate_key_summary = duplicate_key_groups.agg(
    F.count("*").alias("duplicate_key_groups"),
    F.coalesce(F.sum(F.col("rows_per_key") - 1), F.lit(0)).alias("rows_beyond_first"),
)
duplicate_key_summary.show(truncate=False)
duplicate_key_groups.orderBy(F.desc("rows_per_key")).show(20, truncate=False)

+--------------------+-----------------+
|duplicate_key_groups|rows_beyond_first|
+--------------------+-----------------+
|0                   |0                |
+--------------------+-----------------+



+----------+-----------------+-------------------------------+------+----+----------+------------+
|FlightDate|Reporting_Airline|Flight_Number_Reporting_Airline|Origin|Dest|CRSDepTime|rows_per_key|
+----------+-----------------+-------------------------------+------+----+----------+------------+
+----------+-----------------+-------------------------------+------+----+----------+------------+



Business-key repeats are not automatically errors: flight numbers or schedules can occasionally be represented more than once. The displayed examples should be investigated before any later deduplication rule is designed.

## 7. Missing values in important columns

Null and blank-string checks are reported together. Some missingness is expected—for example, completed flights normally have no cancellation code, while delay-cause minutes are populated only for qualifying delayed flights.

In [9]:
important_columns = [
    "FlightDate", "Year", "Month", "DayofMonth", "DayOfWeek",
    "Reporting_Airline", "Flight_Number_Reporting_Airline", "Tail_Number",
    "Origin", "Dest", "CRSDepTime", "CRSArrTime",
    "DepTime", "ArrTime", "Cancelled", "CancellationCode", "Diverted",
    "ArrDelay", "ArrDelayMinutes", "CarrierDelay", "WeatherDelay",
    "NASDelay", "SecurityDelay", "LateAircraftDelay",
]

missing_expressions = []
for column_name in important_columns:
    is_missing = F.col(column_name).isNull() | (F.trim(F.col(column_name).cast("string")) == "")
    missing_expressions.extend([
        F.sum(F.when(is_missing, 1).otherwise(0)).alias(f"{column_name}__missing"),
        F.round(F.avg(F.when(is_missing, 1.0).otherwise(0.0)) * 100, 3).alias(f"{column_name}__missing_pct"),
    ])

flights.agg(*missing_expressions).show(vertical=True, truncate=False)

-RECORD 0------------------------------------------------
 FlightDate__missing                          | 0        
 FlightDate__missing_pct                      | 0.0      
 Year__missing                                | 0        
 Year__missing_pct                            | 0.0      
 Month__missing                               | 0        
 Month__missing_pct                           | 0.0      
 DayofMonth__missing                          | 0        
 DayofMonth__missing_pct                      | 0.0      
 DayOfWeek__missing                           | 0        
 DayOfWeek__missing_pct                       | 0.0      
 Reporting_Airline__missing                   | 0        
 Reporting_Airline__missing_pct               | 0.0      
 Flight_Number_Reporting_Airline__missing     | 1        
 Flight_Number_Reporting_Airline__missing_pct | 0.0      
 Tail_Number__missing                         | 35638    
 Tail_Number__missing_pct                     | 0.256    
 Origin__missi

Interpret missingness in context rather than assuming every null is a defect. This notebook intentionally stops at reporting; cleaning decisions belong in a later data-engineering layer.

## 8. Release temporary cache

The raw DataFrame was cached only to avoid repeated CSV scans during validation. Release it when the checks are complete.

In [10]:
flights.unpersist()
spark.stop()
print("Validation complete; temporary cache released and Spark session stopped.")

Validation complete; temporary cache released and Spark session stopped.
